# CLIP Linear Probe - Kaggle Folder Tiny-GenImage

Notebook này đọc Tiny-GenImage dạng folder trên Kaggle input, không dùng Hugging Face streaming.

Flow:

```text
Kaggle folder images -> frozen CLIP image encoder -> linear classification head -> real/fake
```

CLIP được freeze. Chỉ train tầng cuối bằng mini-batch để giảm RAM.

In [ ]:
%pip install -q open_clip_torch scikit-learn pandas matplotlib tqdm

## 1. Import và tìm code root

In [ ]:
from pathlib import Path
import gc
import json
import sys
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from torch.utils.data import DataLoader
from tqdm.auto import tqdm


def find_code_root():
    candidates = [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]
    kaggle_input = Path("/kaggle/input")
    if kaggle_input.exists():
        candidates.extend(kaggle_input.glob("*"))
        candidates.extend(kaggle_input.glob("*/*"))
        candidates.extend(kaggle_input.glob("*/*/*"))

        for init_file in kaggle_input.rglob("__init__.py"):
            if init_file.parent.name == "data_loader":
                return init_file.parent.parent

    for candidate in candidates:
        if (candidate / "data_loader" / "__init__.py").exists():
            return candidate

    print("/kaggle/input children:")
    if kaggle_input.exists():
        for path in sorted(kaggle_input.glob("*")):
            print(" -", path)
    raise FileNotFoundError("Không tìm thấy data_loader/__init__.py trong Kaggle input. Hãy kiểm tra dataset code có chứa folder HoangHa_Code/data_loader không.")


CODE_ROOT = find_code_root()
PROJECT_ROOT = Path("/kaggle/working") if Path("/kaggle/working").exists() else CODE_ROOT
sys.path.insert(0, str(CODE_ROOT))
print("CODE_ROOT =", CODE_ROOT)
print("PROJECT_ROOT =", PROJECT_ROOT)

from data_loader import (
    TinyGenImageKaggleConfig,
    TinyGenImageKaggleDataset,
    build_kaggle_tiny_index,
    build_kaggle_tiny_splits,
    collate_unified_batch,
    find_tiny_genimage_root,
    summarize_index,
)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("DEVICE =", DEVICE)

## 2. Cấu hình an toàn cho Kaggle RAM

In [ ]:
# Nếu DATASET_ROOT = None, loader sẽ tự tìm Tiny-GenImage trong /kaggle/input.
# Nếu auto-detect sai, set cụ thể, ví dụ:
# DATASET_ROOT = "/kaggle/input/tiny-genimage"
DATASET_ROOT = None

# Chạy một case trước cho chắc RAM. Khi ổn có thể bật RUN_ALL_CASES = True.
RUN_ALL_CASES = False
SELECTED_EXPERIMENT = "combined"

EXPERIMENT_CONFIGS = [
    {"name": "combined", "eval_case": "combined"},
    {"name": "in_domain_biggan", "eval_case": "in_domain", "generator": "BigGAN"},
    {"name": "cross_generator_glide", "eval_case": "cross_generator", "heldout_generator": "GLIDE"},
    {"name": "train_one_generator_biggan", "eval_case": "train_one_generator", "base_generator": "BigGAN"},
]

BALANCE_REAL = True
RANDOM_SEED = 42

# Test nhanh. Đặt None để chạy full, nhưng Kaggle RAM/GPU yếu nên tăng từ từ.
MAX_TRAIN_SAMPLES = 500
MAX_EVAL_SAMPLES = 300

CLIP_MODEL_NAME = "ViT-B-32"
CLIP_PRETRAINED = "openai"
BATCH_SIZE = 16
NUM_WORKERS = 0
NUM_EPOCHS = 3
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4

SAVE_PREDICTIONS = True
SAVE_PLOTS = True
DISPLAY_TABLES = False

OUTPUT_ROOT = PROJECT_ROOT / "outputs" / "clip_linear_probe_kaggle_folder"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
RUN_ID = time.strftime("%Y%m%d_%H%M%S")
print("RUN_ID =", RUN_ID)

## 3. Kiểm tra cấu trúc Tiny-GenImage trên Kaggle

In [ ]:
detected_root = find_tiny_genimage_root(DATASET_ROOT)
print("Detected Tiny-GenImage root:", detected_root)

index_df = build_kaggle_tiny_index(
    TinyGenImageKaggleConfig(dataset_root=str(detected_root))
)
print("Total indexed images:", len(index_df))
print("Generators:", sorted(index_df["generator"].unique().tolist()))
summary = summarize_index(index_df)
display(summary.head(50))

summary.to_csv(OUTPUT_ROOT / f"dataset_structure_summary_{RUN_ID}.csv", index=False)

## 4. Load CLIP frozen encoder

In [ ]:
import open_clip

clip_model, _, clip_preprocess = open_clip.create_model_and_transforms(
    CLIP_MODEL_NAME,
    pretrained=CLIP_PRETRAINED,
    device=DEVICE,
)
clip_model.eval()
for param in clip_model.parameters():
    param.requires_grad = False

print("Loaded CLIP:", CLIP_MODEL_NAME, CLIP_PRETRAINED)

## 5. Model linear head

In [ ]:
class ClipLinearHead(nn.Module):
    def __init__(self, in_dim=512, num_classes=2):
        super().__init__()
        self.fc = nn.Linear(in_dim, num_classes)

    def forward(self, features):
        return self.fc(features)


def encode_clip(images):
    with torch.no_grad():
        features = clip_model.encode_image(images)
        features = features / features.norm(dim=-1, keepdim=True)
        return features.float()


def infer_feature_dim(loader):
    batch = next(iter(loader))
    images = batch["image"].to(DEVICE)
    features = encode_clip(images)
    return int(features.shape[-1])

## 6. Helper train/evaluate

In [ ]:
def seed_everything(seed):
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def make_run_dir(exp):
    run_dir = OUTPUT_ROOT / exp["name"] / RUN_ID
    for subdir in ["checkpoints", "predictions", "metrics", "plots"]:
        (run_dir / subdir).mkdir(parents=True, exist_ok=True)
    return run_dir


def build_splits_for_experiment(exp):
    config = TinyGenImageKaggleConfig(
        dataset_root=str(detected_root),
        eval_case=exp["eval_case"],
        generator=exp.get("generator"),
        heldout_generator=exp.get("heldout_generator"),
        base_generator=exp.get("base_generator"),
        balance_real=BALANCE_REAL,
        seed=RANDOM_SEED,
        max_train_samples=MAX_TRAIN_SAMPLES,
        max_eval_samples=MAX_EVAL_SAMPLES,
    )
    splits = build_kaggle_tiny_splits(config)
    print("\n===", exp["name"], "===")
    print(splits["notes"])
    print("train real/fake:", splits["train_real_count"], splits["train_fake_count"])
    print("eval real/fake:", splits["eval_real_count"], splits["eval_fake_count"])
    return splits


def build_loaders(splits):
    train_dataset = TinyGenImageKaggleDataset(
        splits["train_df"],
        eval_case=splits["eval_case"],
        transform=clip_preprocess,
    )
    eval_dataset = TinyGenImageKaggleDataset(
        splits["eval_df"],
        eval_case=splits["eval_case"],
        transform=clip_preprocess,
    )
    train_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=NUM_WORKERS,
        pin_memory=torch.cuda.is_available(),
        collate_fn=collate_unified_batch,
    )
    eval_loader = DataLoader(
        eval_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=torch.cuda.is_available(),
        collate_fn=collate_unified_batch,
    )
    return train_loader, eval_loader


def train_one_epoch(head, loader, optimizer, criterion):
    head.train()
    losses = []
    for batch in tqdm(loader, desc="train"):
        images = batch["image"].to(DEVICE, non_blocking=True)
        labels = batch["label"].to(DEVICE, non_blocking=True)
        features = encode_clip(images)
        logits = head(features)
        loss = criterion(logits, labels)

        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        optimizer.step()
        losses.append(float(loss.detach().cpu()))
    return float(np.mean(losses)) if losses else 0.0


def predict(head, loader):
    head.eval()
    probs, labels, rows = [], [], []
    with torch.no_grad():
        for batch in tqdm(loader, desc="eval"):
            images = batch["image"].to(DEVICE, non_blocking=True)
            features = encode_clip(images)
            logits = head(features)
            batch_probs = torch.softmax(logits, dim=-1)[:, 1].detach().cpu().numpy()
            probs.extend(batch_probs.tolist())
            labels.extend(batch["label"].numpy().tolist())
            rows.extend(batch["metadata"])
    return np.array(labels, dtype=int), np.array(probs, dtype=float), pd.DataFrame(rows)


def compute_metrics(y_true, y_prob):
    y_pred = (y_prob >= 0.5).astype(int)
    metrics = {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "balanced_accuracy": float(balanced_accuracy_score(y_true, y_pred)),
        "precision": float(precision_score(y_true, y_pred, zero_division=0)),
        "recall": float(recall_score(y_true, y_pred, zero_division=0)),
        "f1": float(f1_score(y_true, y_pred, zero_division=0)),
        "confusion_matrix": confusion_matrix(y_true, y_pred).tolist(),
    }
    if len(np.unique(y_true)) == 2:
        metrics["roc_auc"] = float(roc_auc_score(y_true, y_prob))
        metrics["average_precision"] = float(average_precision_score(y_true, y_prob))
    else:
        metrics["roc_auc"] = None
        metrics["average_precision"] = None
    return metrics


def evaluate_by_generator(pred_df):
    rows = []
    for generator, part in pred_df.groupby("generator"):
        metrics = compute_metrics(part["label"].to_numpy(), part["fake_probability"].to_numpy())
        metrics["generator"] = generator
        metrics["num_samples"] = int(len(part))
        rows.append(metrics)
    return pd.DataFrame(rows).sort_values("generator")

## 7. Helper lưu kết quả

In [ ]:
def save_plots(run_dir, experiment_name, y_true, y_prob, metrics):
    if not SAVE_PLOTS:
        return
    if len(np.unique(y_true)) == 2:
        fpr, tpr, _ = roc_curve(y_true, y_prob)
        fig, ax = plt.subplots(figsize=(5, 4))
        ax.plot(fpr, tpr, label=f'AUROC={metrics["roc_auc"]:.3f}')
        ax.plot([0, 1], [0, 1], linestyle="--")
        ax.set_xlabel("False Positive Rate")
        ax.set_ylabel("True Positive Rate")
        ax.set_title(f"ROC - {experiment_name}")
        ax.legend()
        fig.tight_layout()
        fig.savefig(run_dir / "plots" / "roc_curve.png", dpi=150)
        plt.close(fig)

    cm = np.array(metrics["confusion_matrix"])
    fig, ax = plt.subplots(figsize=(4, 4))
    ax.imshow(cm, cmap="Blues")
    ax.set_xticks([0, 1])
    ax.set_xticklabels(["real", "fake"])
    ax.set_yticks([0, 1])
    ax.set_yticklabels(["real", "fake"])
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")
    ax.set_title(experiment_name)
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(j, i, str(cm[i, j]), ha="center", va="center")
    fig.tight_layout()
    fig.savefig(run_dir / "plots" / "confusion_matrix.png", dpi=150)
    plt.close(fig)


def save_outputs(run_dir, exp, splits, head, y_true, y_prob, meta_df, losses):
    y_pred = (y_prob >= 0.5).astype(int)
    pred_df = meta_df.copy()
    pred_df["label"] = y_true
    pred_df["predicted_label"] = y_pred
    pred_df["fake_probability"] = y_prob
    pred_df["experiment_name"] = exp["name"]
    pred_df["model_name"] = "clip_linear_head_kaggle_folder"

    metrics = compute_metrics(y_true, y_prob)
    metrics.update({
        "experiment_name": exp["name"],
        "eval_case": exp["eval_case"],
        "generator": exp.get("generator"),
        "heldout_generator": exp.get("heldout_generator"),
        "base_generator": exp.get("base_generator"),
        "dataset_root": splits["dataset_root"],
        "balance_real": BALANCE_REAL,
        "max_train_samples": MAX_TRAIN_SAMPLES,
        "max_eval_samples": MAX_EVAL_SAMPLES,
        "clip_model_name": CLIP_MODEL_NAME,
        "clip_pretrained": CLIP_PRETRAINED,
        "batch_size": BATCH_SIZE,
        "num_epochs": NUM_EPOCHS,
        "train_rows": splits["train_rows"],
        "eval_rows": splits["eval_rows"],
        "losses": losses,
        "split_info": {key: value for key, value in splits.items() if key not in {"train_df", "eval_df", "full_index"}},
    })
    generator_metrics_df = evaluate_by_generator(pred_df)

    torch.save(head.state_dict(), run_dir / "checkpoints" / "linear_head.pt")
    if SAVE_PREDICTIONS:
        pred_df.to_csv(run_dir / "predictions" / "predictions.csv", index=False)
    generator_metrics_df.to_csv(run_dir / "metrics" / "generator_metrics.csv", index=False)
    with open(run_dir / "metrics" / "overall_metrics.json", "w", encoding="utf-8") as f:
        json.dump(metrics, f, ensure_ascii=False, indent=2)
    save_plots(run_dir, exp["name"], y_true, y_prob, metrics)
    return metrics, generator_metrics_df

## 8. Chạy experiment

In [ ]:
def run_experiment(exp):
    seed_everything(RANDOM_SEED)
    run_dir = make_run_dir(exp)
    splits = build_splits_for_experiment(exp)
    splits["train_df"].to_csv(run_dir / "metrics" / "train_split.csv", index=False)
    splits["eval_df"].to_csv(run_dir / "metrics" / "eval_split.csv", index=False)

    train_loader, eval_loader = build_loaders(splits)
    feature_dim = infer_feature_dim(train_loader)
    head = ClipLinearHead(in_dim=feature_dim, num_classes=2).to(DEVICE)
    optimizer = torch.optim.AdamW(head.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    criterion = nn.CrossEntropyLoss()

    losses = []
    for epoch in range(NUM_EPOCHS):
        loss = train_one_epoch(head, train_loader, optimizer, criterion)
        losses.append(loss)
        print(f"epoch {epoch + 1}/{NUM_EPOCHS} loss={loss:.4f}")

    y_true, y_prob, meta_df = predict(head, eval_loader)
    metrics, generator_metrics_df = save_outputs(run_dir, exp, splits, head, y_true, y_prob, meta_df, losses)
    print("Done:", exp["name"])
    print("RUN_DIR:", run_dir)
    print(json.dumps(metrics, ensure_ascii=False, indent=2))
    if DISPLAY_TABLES:
        display(generator_metrics_df)

    del train_loader, eval_loader, head, optimizer, criterion, y_true, y_prob, meta_df
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return metrics


experiments_to_run = EXPERIMENT_CONFIGS if RUN_ALL_CASES else [exp for exp in EXPERIMENT_CONFIGS if exp["name"] == SELECTED_EXPERIMENT]
if not experiments_to_run:
    raise ValueError(f"SELECTED_EXPERIMENT={SELECTED_EXPERIMENT!r} không có trong EXPERIMENT_CONFIGS")

all_metrics = []
for exp in experiments_to_run:
    all_metrics.append(run_experiment(exp))

summary_df = pd.DataFrame(all_metrics)
summary_path = OUTPUT_ROOT / f"summary_metrics_{RUN_ID}.csv"
summary_df.to_csv(summary_path, index=False)
print("Summary saved:", summary_path)
display(summary_df[["experiment_name", "eval_case", "accuracy", "balanced_accuracy", "precision", "recall", "f1", "roc_auc", "average_precision", "train_rows", "eval_rows"]])

## 9. Ghi chú

- Notebook này đọc ảnh từ Kaggle input local, không dùng Hugging Face streaming.
- Mặc định chỉ chạy một case để tránh Kaggle RAM restart.
- Khi muốn chạy cả 4 case, đặt `RUN_ALL_CASES = True`.
- Khi muốn chạy full dữ liệu, đặt `MAX_TRAIN_SAMPLES = None` và `MAX_EVAL_SAMPLES = None`, nhưng nên tăng từ từ.